In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
# Load cleaned production dataset

production_df = pd.read_csv(
    "../data/processed/PhiUSIIL_production.csv"
)

print("Dataset shape:", production_df.shape)
print("Missing values:", production_df.isnull().sum().sum())
print("\nLabel distribution:")
print(production_df["label"].value_counts())

Dataset shape: (235370, 19)
Missing values: 0

Label distribution:
label
1    134850
0    100520
Name: count, dtype: int64


In [3]:
# Load cleaned production dataset

production_df = pd.read_csv(
    "../data/processed/PhiUSIIL_production.csv"
)

print("Dataset shape:", production_df.shape)
print("Missing values:", production_df.isnull().sum().sum())
print("\nLabel distribution:")
print(production_df["label"].value_counts())

Dataset shape: (235370, 19)
Missing values: 0

Label distribution:
label
1    134850
0    100520
Name: count, dtype: int64


In [4]:
# Cell 4 - Domain-aware train/holdout split

from urllib.parse import urlparse

def get_domain(url):
    try:
        return urlparse(str(url)).hostname or ""
    except:
        return ""

production_df["DomainGroup"] = production_df["URL"].apply(get_domain)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, holdout_idx = next(
    splitter.split(
        production_df,
        production_df["label"],
        groups=production_df["DomainGroup"]
    )
)

domain_train_df = production_df.iloc[train_idx].copy()
domain_holdout_df = production_df.iloc[holdout_idx].copy()

overlap = set(domain_train_df["DomainGroup"]) & set(domain_holdout_df["DomainGroup"])

print("Training rows:", len(domain_train_df))
print("Holdout rows:", len(domain_holdout_df))
print("Domain overlap:", len(overlap))
print("\nTraining labels:")
print(domain_train_df["label"].value_counts())

Training rows: 187856
Holdout rows: 47514
Domain overlap: 0

Training labels:
label
1    107828
0     80028
Name: count, dtype: int64


In [5]:
# Cell 5 - Legitimate URLs from training split ONLY

legitimate_train_urls = (
    domain_train_df.loc[
        domain_train_df["label"] == 1,
        "URL"
    ]
    .dropna()
    .astype(str)
    .tolist()
)

print("Legitimate training URLs:", len(legitimate_train_urls))

Legitimate training URLs: 107828


In [6]:
# Cell 6 - Generate structurally diverse legitimate URLs

STRUCTURAL_PATHS = [
    "/",

    # Depth 1
    "/about",
    "/contact",
    "/products",
    "/services",
    "/blog",
    "/help",
    "/news",
    "/docs",

    # Depth 2
    "/products/item",
    "/blog/article",
    "/news/latest",
    "/docs/guide",
    "/support/account",
    "/company/about",
    "/resources/articles",

    # Depth 3
    "/products/category/item",
    "/docs/guide/setup",
    "/support/account/settings",
    "/resources/articles/latest",
    "/company/about/team",

    # Depth 4
    "/docs/guide/setup/windows",
    "/products/category/item/details",
    "/support/account/settings/privacy",
]

structural_legitimate_urls = []

for original_url in legitimate_train_urls:

    base_url = original_url.rstrip("/")

    variants = [base_url]

    if "://www." in base_url:
        variants.append(
            base_url.replace("://www.", "://", 1)
        )

    for variant in variants:
        for path in STRUCTURAL_PATHS:
            structural_legitimate_urls.append(
                variant + path
            )

print("Legitimate source URLs:", len(legitimate_train_urls))
print("Structural pool:", len(structural_legitimate_urls))

Legitimate source URLs: 107828
Structural pool: 5175744


In [7]:
# Cell 7 - Controlled 100k augmentation sample

structural_sample = pd.Series(
    structural_legitimate_urls
).sample(
    n=100000,
    random_state=42
).tolist()

print("Total structural pool:", len(structural_legitimate_urls))
print("Sampled for augmentation:", len(structural_sample))

Total structural pool: 5175744
Sampled for augmentation: 100000


In [8]:
# Cell 8 - Use the same feature extractor as the application

import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.src.feature_extractor import extract_url_features

print("Feature extractor imported successfully.")

Feature extractor imported successfully.


In [9]:
# Cell 9 - Extract features from augmented legitimate URLs

augmented_features_df = pd.DataFrame(
    [
        extract_url_features(url)
        for url in structural_sample
    ]
)

augmented_features_df.insert(
    0,
    "URL",
    structural_sample
)

augmented_features_df["label"] = 1

print("Augmented shape:", augmented_features_df.shape)

print("\nNoOfSlashes:")
print(
    augmented_features_df["NoOfSlashes"]
    .value_counts()
    .sort_index()
)

print("\nPathDepth:")
print(
    augmented_features_df["PathDepth"]
    .value_counts()
    .sort_index()
)

Augmented shape: (100000, 19)

NoOfSlashes:
NoOfSlashes
3    37379
4    29139
5    20942
6    12540
Name: count, dtype: int64

PathDepth:
PathDepth
0     4199
1    33180
2    29139
3    20942
4    12540
Name: count, dtype: int64


In [10]:
# Cell 10 - Final augmented training dataset

final_train_df = pd.concat(
    [
        domain_train_df,
        augmented_features_df
    ],
    ignore_index=True
)

X_train_final = final_train_df[ROBUSTNESS_FEATURES].copy()
y_train_final = final_train_df["label"].copy()

X_holdout = domain_holdout_df[ROBUSTNESS_FEATURES].copy()
y_holdout = domain_holdout_df["label"].copy()

print("Final training rows:", len(final_train_df))
print("Holdout rows:", len(domain_holdout_df))
print("Feature count:", X_train_final.shape[1])
print("Missing training values:", X_train_final.isnull().sum().sum())

print("\nFinal training labels:")
print(y_train_final.value_counts())

NameError: name 'ROBUSTNESS_FEATURES' is not defined

In [ ]:
print("ROBUSTNESS_FEATURES exists:", "ROBUSTNESS_FEATURES" in globals())

ROBUSTNESS_FEATURES exists: False


In [ ]:
ROBUSTNESS_FEATURES = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLD",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "IsHTTPS",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth",
]

print("Feature count:", len(ROBUSTNESS_FEATURES))

Feature count: 17


In [ ]:
# Cell 11 - Train structural-fix candidate model

tld_encoder = TargetEncoder(
    target_type="binary",
    random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "tld_target_encoder",
            tld_encoder,
            ["TLD"]
        ),
        (
            "numeric",
            "passthrough",
            [
                feature
                for feature in ROBUSTNESS_FEATURES
                if feature != "TLD"
            ]
        )
    ],
    remainder="drop"
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

structural_candidate_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

structural_candidate_pipeline.fit(
    X_train_final,
    y_train_final
)

print("Candidate model trained successfully.")
print("Classes:", structural_candidate_pipeline.classes_)

NameError: name 'X_train_final' is not defined

In [ ]:
print("ROBUSTNESS_FEATURES exists:", "ROBUSTNESS_FEATURES" in globals())

ROBUSTNESS_FEATURES exists: False


In [ ]:
# Cell 12 - Evaluate on untouched domain-aware holdout

holdout_predictions = structural_candidate_pipeline.predict(
    X_holdout
)

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_holdout,
            holdout_predictions
        ),
        4
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_holdout,
        holdout_predictions
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_holdout,
        holdout_predictions
    )
)

Accuracy: 0.9857

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     20492
           1       0.98      1.00      0.99     27022

    accuracy                           0.99     47514
   macro avg       0.99      0.98      0.99     47514
weighted avg       0.99      0.99      0.99     47514

Confusion Matrix:
[[19877   615]
 [   64 26958]]


In [ ]:
# Cell 13 - Legitimate URL sanity test

legitimate_test_urls = [
    "https://www.google.com",
    "https://www.google.com/maps",
    "https://www.google.com/maps/place",
    "https://www.google.com/a/b",
    "https://www.google.com/a/b/c",
    "https://www.paypal.com",
    "https://www.paypal.com/home",
    "https://www.paypal.com/in/home",
]

classes = list(structural_candidate_pipeline.classes_)

phishing_index = classes.index(0)
legitimate_index = classes.index(1)

results = []

for url in legitimate_test_urls:

    features = extract_url_features(url)

    input_df = pd.DataFrame(
        [features]
    )[ROBUSTNESS_FEATURES]

    prediction = structural_candidate_pipeline.predict(
        input_df
    )[0]

    probabilities = structural_candidate_pipeline.predict_proba(
        input_df
    )[0]

    results.append({
        "URL": url,
        "Prediction": (
            "Legitimate"
            if prediction == 1
            else "Phishing"
        ),
        "Legitimate %": round(
            probabilities[legitimate_index] * 100,
            2
        ),
        "Phishing Risk %": round(
            probabilities[phishing_index] * 100,
            2
        ),
    })

pd.DataFrame(results)

,URL,Prediction,Legitimate %,Phishing Risk %
0,https://www.google.com,Legitimate,99.43,0.57
1,https://www.google.com/maps,Legitimate,100.00,0.00
2,https://www.google.com/maps/place,Legitimate,100.00,0.00
3,https://www.google.com/a/b,Legitimate,79.00,21.00
4,https://www.google.com/a/b/c,Legitimate,63.33,36.67
5,https://www.paypal.com,Legitimate,99.43,0.57
6,https://www.paypal.com/home,Legitimate,100.00,0.00
7,https://www.paypal.com/in/home,Legitimate,87.33,12.67


In [ ]:
# Cell 14 - Phishing-like URL sanity test

phishing_test_urls = [
    "https://paypal-login-security.com",
    "https://account-verification-update.com/login",
    "https://secure-bank-login.xyz/account",
    "https://verify-your-account.com/update",
    "https://login-security-check.net/verify",
]

results = []

for url in phishing_test_urls:

    features = extract_url_features(url)

    input_df = pd.DataFrame(
        [features]
    )[ROBUSTNESS_FEATURES]

    prediction = structural_candidate_pipeline.predict(
        input_df
    )[0]

    probabilities = structural_candidate_pipeline.predict_proba(
        input_df
    )[0]

    results.append({
        "URL": url,
        "Prediction": (
            "Legitimate"
            if prediction == 1
            else "Phishing"
        ),
        "Legitimate %": round(
            probabilities[legitimate_index] * 100,
            2
        ),
        "Phishing Risk %": round(
            probabilities[phishing_index] * 100,
            2
        ),
    })

pd.DataFrame(results)

c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\Tr

,URL,Prediction,Legitimate %,Phishing Risk %
0,https://paypal-login-security.com,Phishing,1.72,98.28
1,https://account-verification-update.com/login,Phishing,33.33,66.67
2,https://secure-bank-login.xyz/account,Phishing,3.00,97.00
3,https://verify-your-account.com/update,Phishing,18.00,82.00
4,https://login-security-check.net/verify,Phishing,13.67,86.33


In [ ]:
# Cell 15 - Save validated production model safely

from pathlib import Path
import shutil
import joblib

model_path = Path(
    "../models/trustlens_production_pipeline.joblib"
)

backup_path = Path(
    "../models/trustlens_production_pipeline_backup.joblib"
)

# Preserve existing production model if backup does not already exist
if model_path.exists() and not backup_path.exists():
    shutil.copy2(model_path, backup_path)
    print("Backup created.")
elif backup_path.exists():
    print("Backup already exists - leaving it untouched.")

# Save validated structural-fix model
joblib.dump(
    structural_candidate_pipeline,
    model_path
)

print("New production model saved successfully.")
print("Saved to:", model_path)

Backup already exists - leaving it untouched.
New production model saved successfully.
Saved to: ..\models\trustlens_production_pipeline.joblib


In [ ]:
# Cell 16 - Final reload verification

reloaded_pipeline = joblib.load(
    "../models/trustlens_production_pipeline.joblib"
)

print("Production model reloaded successfully.")
print("Classes:", reloaded_pipeline.classes_)

test_url = "https://www.google.com/maps/place"

features = extract_url_features(test_url)
input_df = pd.DataFrame([features])[ROBUSTNESS_FEATURES]

prediction = reloaded_pipeline.predict(input_df)[0]
probabilities = reloaded_pipeline.predict_proba(input_df)[0]

classes = list(reloaded_pipeline.classes_)
phishing_index = classes.index(0)

print("Test URL:", test_url)
print(
    "Prediction:",
    "Legitimate" if prediction == 1 else "Phishing"
)
print(
    "Phishing Risk:",
    round(probabilities[phishing_index] * 100, 2),
    "%"
)

c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


Production model reloaded successfully.
Classes: [0 1]


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\Tr

Test URL: https://www.google.com/maps/place
Prediction: Legitimate
Phishing Risk: 0.0 %


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\swana\OneDrive\Desktop\Tr

In [ ]:
print("\nPathDepth:")
print(
    augmented_features_df["PathDepth"]
    .value_counts()
    .sort_index()
)


PathDepth:
PathDepth
0     4199
1    33180
2    29139
3    20942
4    12540
Name: count, dtype: int64


In [ ]:
# =========================================================
# QUERY-HEAVY LEGITIMATE URL AUGMENTATION
# =========================================================

import random
from urllib.parse import urlsplit, urlunsplit

random.seed(42)

QUERY_TEMPLATES = [
    "/search?q=example",
    "/search?q=example&page=2",
    "/products?id=12345",
    "/article?id=987654",
    "/account/settings?tab=privacy",
    "/docs/view?id=12345&lang=en",
    "/results?query=example&sort=recent",
    "/location?lat=19.1922063&lng=72.9234965",
    "/maps/@19.1922063,72.9234965,11z",
    "/maps/@19.1922063,72.9234965,11z?entry=ttu",
    "/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view",
    "/redirect?next=%2Faccount%2Fsettings",
    "/search?q=hello%20world",
    "/docs?page=2&section=installation",
    "/view?item=123456&source=web&lang=en",
]

query_legitimate_urls = []

for original_url in legitimate_train_urls:

    try:
        parsed = urlsplit(original_url)

        if not parsed.scheme or not parsed.netloc:
            continue

        # IMPORTANT:
        # use only scheme + legitimate training hostname
        # instead of appending onto the original path/query
        base_url = urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                "",
                "",
                "",
            )
        ).rstrip("/")

        variants = [base_url]

        # www/non-www structural variation
        if "://www." in base_url:
            variants.append(
                base_url.replace(
                    "://www.",
                    "://",
                    1,
                )
            )

        for variant in variants:

            for template in QUERY_TEMPLATES:

                query_legitimate_urls.append(
                    variant + template
                )

    except Exception:
        continue


print(
    "Query-heavy legitimate pool:",
    len(query_legitimate_urls)
)

Query-heavy legitimate pool: 3234840


In [ ]:
# =========================================================
# SAMPLE QUERY-HEAVY LEGITIMATE URLS
# =========================================================

QUERY_AUGMENTATION_SIZE = 50_000

query_augmented_urls = random.sample(
    query_legitimate_urls,
    k=min(
        QUERY_AUGMENTATION_SIZE,
        len(query_legitimate_urls),
    ),
)

print(
    "Query-heavy URLs sampled:",
    len(query_augmented_urls),
)

print("\nExamples:")

for url in query_augmented_urls[:10]:
    print(url)

Query-heavy URLs sampled: 50000

Examples:
https://www.rainforest-alliance.org/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view
https://www.wszia.opole.pl/results?query=example&sort=recent
https://emilyrosetheartist.com/location?lat=19.1922063&lng=72.9234965
https://www.marasgroup.com.au/maps/@19.1922063,72.9234965,11z
https://eparhija-sremska.rs/redirect?next=%2Faccount%2Fsettings
https://www.intelligence.gov/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view
https://www.poshhaus.com/article?id=987654
https://theafiyacenter.org/maps/@19.1922063,72.9234965,11z?entry=ttu
https://www.collegefashion.net/account/settings?tab=privacy
https://coloradoheliops.com/maps/@19.1922063,72.9234965,11z?entry=ttu&mode=view


In [ ]:
# =========================================================
# EXTRACT FEATURES FROM QUERY-HEAVY LEGITIMATE URLS
# =========================================================

# Exact 17 production features
ROBUSTNESS_FEATURES = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLD",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "IsHTTPS",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth",
]


# Extract the same features used by TrustLens
query_augmented_features_df = pd.DataFrame(
    [
        extract_url_features(url)
        for url in query_augmented_urls
    ]
)

# Add original URL for inspection
query_augmented_features_df.insert(
    0,
    "URL",
    query_augmented_urls,
)

# All generated URLs are legitimate examples
query_augmented_features_df["label"] = 1


# =========================================================
# BASIC CHECKS
# =========================================================

print(
    "Query augmentation shape:",
    query_augmented_features_df.shape,
)

print(
    "Feature count:",
    len(ROBUSTNESS_FEATURES),
)

print(
    "Missing values:",
    query_augmented_features_df[
        ROBUSTNESS_FEATURES
    ].isnull().sum().sum(),
)


# =========================================================
# DISTRIBUTION CHECKS
# =========================================================

print("\nURL Length:")
print(
    query_augmented_features_df[
        "URLLength"
    ].describe()
)

print("\nDigits:")
print(
    query_augmented_features_df[
        "NoOfDegitsInURL"
    ].describe()
)

print("\nEquals signs:")
print(
    query_augmented_features_df[
        "NoOfEqualsInURL"
    ]
    .value_counts()
    .sort_index()
)

print("\nQuestion marks:")
print(
    query_augmented_features_df[
        "NoOfQMarkInURL"
    ]
    .value_counts()
    .sort_index()
)

print("\nObfuscation:")
print(
    query_augmented_features_df[
        "HasObfuscation"
    ]
    .value_counts()
    .sort_index()
)

Query augmentation shape: (50000, 19)
Feature count: 17
Missing values: 0

URL Length:
count    50000.000000
mean        55.879340
std         10.864878
min         30.000000
25%         48.000000
50%         56.000000
75%         63.000000
max         98.000000
Name: URLLength, dtype: float64

Digits:
count    50000.00000
mean         7.13880
std          7.77661
min          0.00000
25%          1.00000
50%          5.00000
75%         18.00000
max         25.00000
Name: NoOfDegitsInURL, dtype: float64

Equals signs:
NoOfEqualsInURL
0     3342
1    23350
2    19913
3     3395
Name: count, dtype: int64

Question marks:
NoOfQMarkInURL
0     3342
1    46658
Name: count, dtype: int64

Obfuscation:
HasObfuscation
0    43407
1     6593
Name: count, dtype: int64


In [ ]:
# =========================================================
# FINAL TRAINING DATA
# Original + Path Augmentation + Query Augmentation
# =========================================================

final_train_df_v2 = pd.concat(
    [
        domain_train_df,
        augmented_features_df,
        query_augmented_features_df,
    ],
    ignore_index=True,
)


X_train_final_v2 = final_train_df_v2[
    ROBUSTNESS_FEATURES
].copy()

y_train_final_v2 = final_train_df_v2[
    "label"
].copy()


# Holdout remains completely untouched
X_holdout_v2 = domain_holdout_df[
    ROBUSTNESS_FEATURES
].copy()

y_holdout_v2 = domain_holdout_df[
    "label"
].copy()


print(
    "Final training rows:",
    len(final_train_df_v2)
)

print(
    "Holdout rows:",
    len(domain_holdout_df)
)

print(
    "Feature count:",
    X_train_final_v2.shape[1]
)

print(
    "Missing training values:",
    X_train_final_v2.isnull().sum().sum()
)

print("\nFinal training labels:")
print(
    y_train_final_v2.value_counts()
)

Final training rows: 337856
Holdout rows: 47514
Feature count: 17
Missing training values: 0

Final training labels:
label
1    257828
0     80028
Name: count, dtype: int64


In [ ]:
# =========================================================
# TRAIN V2 CANDIDATE MODEL
# DO NOT SAVE TO PRODUCTION YET
# =========================================================

categorical_features_v2 = ["TLD"]

numeric_features_v2 = [
    feature
    for feature in ROBUSTNESS_FEATURES
    if feature != "TLD"
]

preprocessor_v2 = ColumnTransformer(
    transformers=[
        (
            "tld",
            TargetEncoder(
                target_type="binary",
                random_state=42,
            ),
            categorical_features_v2,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features_v2,
        ),
    ]
)

candidate_model_v2 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

structural_candidate_pipeline_v2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2),
        ("model", candidate_model_v2),
    ]
)


print("Training V2 candidate...")

structural_candidate_pipeline_v2.fit(
    X_train_final_v2,
    y_train_final_v2,
)

print("V2 candidate training complete.")

Training V2 candidate...


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V2 candidate training complete.


In [ ]:
# =========================================================
# EVALUATE V2 CANDIDATE ON UNTOUCHED HOLDOUT
# =========================================================

holdout_predictions_v2 = structural_candidate_pipeline_v2.predict(
    X_holdout_v2
)

holdout_accuracy_v2 = accuracy_score(
    y_holdout_v2,
    holdout_predictions_v2,
)

print(
    "V2 Holdout Accuracy:",
    round(holdout_accuracy_v2, 4)
)

print("\nClassification Report:")
print(
    classification_report(
        y_holdout_v2,
        holdout_predictions_v2,
        digits=4,
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_holdout_v2,
        holdout_predictions_v2,
    )
)

V2 Holdout Accuracy: 0.9853

Classification Report:
              precision    recall  f1-score   support

           0     0.9951    0.9706    0.9827     20492
           1     0.9781    0.9964    0.9872     27022

    accuracy                         0.9853     47514
   macro avg     0.9866    0.9835    0.9850     47514
weighted avg     0.9855    0.9853    0.9853     47514


Confusion Matrix:
[[19890   602]
 [   97 26925]]


In [ ]:
# =========================================================
# V2 LEGITIMATE URL SANITY TESTS
# =========================================================

legitimate_test_urls_v2 = [
    # Original simple/path tests
    "https://www.google.com",
    "https://www.google.com/maps",
    "https://www.google.com/maps/place",
    "https://www.google.com/a/b",
    "https://www.google.com/a/b/c",

    "https://www.paypal.com",
    "https://www.paypal.com/home",
    "https://www.paypal.com/in/home",

    # Real long Google Maps URL that previously failed
    "https://www.google.com/maps/@19.1922063,72.9234965,11.82z?entry=ttu&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D",

    # Generic legitimate-looking long structures
    "https://example.com/search?q=hello%20world&page=2",
    "https://example.com/location?lat=19.1922063&lng=72.9234965",
    "https://example.com/docs/view?id=12345&lang=en",
]


def test_candidate_url_v2(url):

    features = pd.DataFrame(
        [extract_url_features(url)]
    )[ROBUSTNESS_FEATURES]

    probabilities = (
        structural_candidate_pipeline_v2
        .predict_proba(features)[0]
    )

    classes = (
        structural_candidate_pipeline_v2
        .classes_
    )

    probability_map = dict(
        zip(classes, probabilities)
    )

    phishing_risk = (
        probability_map.get(0, 0.0) * 100
    )

    legitimate_probability = (
        probability_map.get(1, 0.0) * 100
    )

    prediction = (
        "Legitimate"
        if legitimate_probability >= phishing_risk
        else "Phishing"
    )

    print(url)
    print(
        f"Prediction: {prediction} | "
        f"Legit: {legitimate_probability:.2f}% | "
        f"Phishing risk: {phishing_risk:.2f}%"
    )
    print("-" * 90)


for url in legitimate_test_urls_v2:
    test_candidate_url_v2(url)

https://www.google.com
Prediction: Legitimate | Legit: 100.00% | Phishing risk: 0.00%
------------------------------------------------------------------------------------------
https://www.google.com/maps
Prediction: Legitimate | Legit: 100.00% | Phishing risk: 0.00%
------------------------------------------------------------------------------------------
https://www.google.com/maps/place
Prediction: Legitimate | Legit: 99.67% | Phishing risk: 0.33%
------------------------------------------------------------------------------------------
https://www.google.com/a/b
Prediction: Legitimate | Legit: 81.00% | Phishing risk: 19.00%
------------------------------------------------------------------------------------------
https://www.google.com/a/b/c
Prediction: Legitimate | Legit: 63.00% | Phishing risk: 37.00%
------------------------------------------------------------------------------------------
https://www.paypal.com
Prediction: Legitimate | Legit: 100.00% | Phishing risk: 0.00%
----

In [ ]:
# =========================================================
# DIAGNOSE THE REAL LONG LEGITIMATE URL
# =========================================================

problem_url = (
    "https://www.google.com/maps/"
    "@19.1922063,72.9234965,11.82z"
    "?entry=ttu"
    "&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D"
)

problem_features = extract_url_features(problem_url)

print("Problem URL length:", len(problem_url))
print("\nProblem URL features:\n")

for feature in ROBUSTNESS_FEATURES:
    print(
        f"{feature:28} : "
        f"{problem_features[feature]}"
    )


print("\n" + "=" * 60)
print("QUERY AUGMENTATION COMPARISON")
print("=" * 60)

numeric_compare_features = [
    "URLLength",
    "DomainLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth",
]

for feature in numeric_compare_features:

    series = query_augmented_features_df[feature]

    print(f"\n{feature}")
    print(
        "Problem:",
        problem_features[feature],
        "| Aug mean:",
        round(series.mean(), 2),
        "| 95%:",
        round(series.quantile(0.95), 2),
        "| Max:",
        series.max(),
    )

Problem URL length: 109

Problem URL features:

URLLength                    : 109
DomainLength                 : 14
IsDomainIP                   : 0
TLD                          : com
TLDLength                    : 3
NoOfSubDomain                : 1
HasObfuscation               : 1
NoOfObfuscatedChar           : 6
NoOfDegitsInURL              : 26
NoOfEqualsInURL              : 2
NoOfQMarkInURL               : 1
IsHTTPS                      : 1
NoOfDots                     : 5
NoOfSlashes                  : 4
SuspiciousKeywordCount       : 0
HasHyphenInDomain            : 0
PathDepth                    : 2

QUERY AUGMENTATION COMPARISON

URLLength
Problem: 109 | Aug mean: 55.88 | 95%: 75.0 | Max: 98

DomainLength
Problem: 14 | Aug mean: 17.23 | 95%: 26.0 | Max: 45

NoOfSubDomain
Problem: 1 | Aug mean: 0.66 | 95%: 2.0 | Max: 4

HasObfuscation
Problem: 1 | Aug mean: 0.13 | 95%: 1.0 | Max: 1

NoOfObfuscatedChar
Problem: 6 | Aug mean: 0.6 | 95%: 6.0 | Max: 6

NoOfDegitsInURL
Problem: 26 |

In [ ]:
# =========================================================
# V3 - BETTER QUERY AUGMENTATION
# Normal + Long/Complex Legitimate URL Structures
# =========================================================

import random
from urllib.parse import urlsplit, urlunsplit

random.seed(42)


NORMAL_QUERY_TEMPLATES = [
    "/search?q=example",
    "/search?q=example&page=2",
    "/products?id=12345",
    "/article?id=987654",
    "/account/settings?tab=privacy",
    "/docs/view?id=12345&lang=en",
    "/results?query=example&sort=recent",
    "/location?lat=19.1922063&lng=72.9234965",
    "/search?q=hello%20world",
    "/redirect?next=%2Faccount%2Fsettings",
]


COMPLEX_QUERY_TEMPLATES = [
    # Coordinates + queries
    "/location/@19.1922063,72.9234965,11.82z"
    "?entry=web&mode=view",

    "/location/@19.1922063,72.9234965,11.82z"
    "?entry=web&source=browser&mode=view",

    # Coordinates + encoded parameter
    "/location/@19.1922063,72.9234965,11.82z"
    "?entry=web&data=ExampleLongValue1234567890%3D%3D",

    "/location/@19.1922063,72.9234965,11.82z"
    "?source=browser&data=AbCdEf12345678901234567890%3D%3D",

    # Long IDs / tokens
    "/view/item/1234567890"
    "?session=ABCDEF12345678901234567890&lang=en",

    "/docs/view"
    "?id=12345678901234567890"
    "&source=browser"
    "&lang=en",

    # Encoded values
    "/redirect"
    "?next=%2Faccount%2Fsettings%2Fprivacy"
    "&source=web",

    "/search"
    "?q=hello%20world"
    "&category=articles"
    "&page=123456",

    # Decimal-heavy structures
    "/data/@40.712776,-74.005974,12.50z"
    "?view=detail&source=web",

    "/place/@51.507351,-0.127758,14.25z"
    "?entry=browser"
    "&data=Example1234567890%3D%3D",
]


better_query_pool = []

for original_url in legitimate_train_urls:

    try:
        parsed = urlsplit(original_url)

        if not parsed.scheme or not parsed.netloc:
            continue

        base_url = urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                "",
                "",
                "",
            )
        ).rstrip("/")

        variants = [base_url]

        if "://www." in base_url:
            variants.append(
                base_url.replace(
                    "://www.",
                    "://",
                    1,
                )
            )

        for variant in variants:

            for template in NORMAL_QUERY_TEMPLATES:
                better_query_pool.append(
                    variant + template
                )

            for template in COMPLEX_QUERY_TEMPLATES:
                better_query_pool.append(
                    variant + template
                )

    except Exception:
        continue


print(
    "Better query pool:",
    len(better_query_pool)
)

Better query pool: 4313120


In [ ]:
# =========================================================
# V3 - STRATIFIED QUERY AUGMENTATION SAMPLE
# 25k normal + 25k complex
# =========================================================

normal_query_pool = []
complex_query_pool = []

for original_url in legitimate_train_urls:

    try:
        parsed = urlsplit(original_url)

        if not parsed.scheme or not parsed.netloc:
            continue

        base_url = urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                "",
                "",
                "",
            )
        ).rstrip("/")

        variants = [base_url]

        if "://www." in base_url:
            variants.append(
                base_url.replace(
                    "://www.",
                    "://",
                    1,
                )
            )

        for variant in variants:

            for template in NORMAL_QUERY_TEMPLATES:
                normal_query_pool.append(
                    variant + template
                )

            for template in COMPLEX_QUERY_TEMPLATES:
                complex_query_pool.append(
                    variant + template
                )

    except Exception:
        continue


NORMAL_SAMPLE_SIZE = 25_000
COMPLEX_SAMPLE_SIZE = 25_000

normal_query_sample = random.sample(
    normal_query_pool,
    k=min(
        NORMAL_SAMPLE_SIZE,
        len(normal_query_pool),
    ),
)

complex_query_sample = random.sample(
    complex_query_pool,
    k=min(
        COMPLEX_SAMPLE_SIZE,
        len(complex_query_pool),
    ),
)

better_query_sample = (
    normal_query_sample
    + complex_query_sample
)

random.shuffle(
    better_query_sample
)


print(
    "Normal query pool:",
    len(normal_query_pool)
)

print(
    "Complex query pool:",
    len(complex_query_pool)
)

print(
    "Normal sampled:",
    len(normal_query_sample)
)

print(
    "Complex sampled:",
    len(complex_query_sample)
)

print(
    "Total V3 query sample:",
    len(better_query_sample)
)

print("\nExamples:")

for url in better_query_sample[:10]:
    print(url)

Normal query pool: 2156560
Complex query pool: 2156560
Normal sampled: 25000
Complex sampled: 25000
Total V3 query sample: 50000

Examples:
https://www.maroc-hebdo.press.ma/search?q=hello%20world
https://koama.es/search?q=example&page=2
https://wolfgangdigital.com/location/@19.1922063,72.9234965,11.82z?entry=web&mode=view
https://www.tshirtsthatsuck.com/products?id=12345
https://www.existentialcomics.com/results?query=example&sort=recent
https://www.neoformix.com/search?q=example&page=2
https://www.lugaresdenieve.com/docs/view?id=12345678901234567890&source=browser&lang=en
https://elgincounty.ca/location/@19.1922063,72.9234965,11.82z?source=browser&data=AbCdEf12345678901234567890%3D%3D
https://orient-watch.jp/search?q=example
https://www.elisgeo.com/location/@19.1922063,72.9234965,11.82z?source=browser&data=AbCdEf12345678901234567890%3D%3D


In [ ]:
# =========================================================
# V3 - EXTRACT FEATURES + COVERAGE CHECK
# =========================================================

better_query_features_df = pd.DataFrame(
    [
        extract_url_features(url)
        for url in better_query_sample
    ]
)

better_query_features_df.insert(
    0,
    "URL",
    better_query_sample
)

better_query_features_df["label"] = 1


print(
    "V3 query augmentation shape:",
    better_query_features_df.shape
)

print(
    "Missing values:",
    better_query_features_df[
        ROBUSTNESS_FEATURES
    ].isnull().sum().sum()
)


print("\nURL Length:")
print(
    better_query_features_df[
        "URLLength"
    ].describe()
)

print("\nDigits:")
print(
    better_query_features_df[
        "NoOfDegitsInURL"
    ].describe()
)

print("\nObfuscated characters:")
print(
    better_query_features_df[
        "NoOfObfuscatedChar"
    ].describe()
)

print("\nDots:")
print(
    better_query_features_df[
        "NoOfDots"
    ].describe()
)


print("\n" + "=" * 60)
print("PROBLEM URL VS V3 COVERAGE")
print("=" * 60)

coverage_features = [
    "URLLength",
    "NoOfDegitsInURL",
    "NoOfObfuscatedChar",
    "NoOfDots",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfSlashes",
    "PathDepth",
]

for feature in coverage_features:

    series = better_query_features_df[feature]

    print(f"\n{feature}")
    print(
        "Problem:",
        problem_features[feature],
        "| V3 mean:",
        round(series.mean(), 2),
        "| V3 95%:",
        round(series.quantile(0.95), 2),
        "| V3 max:",
        series.max(),
    )

V3 query augmentation shape: (50000, 19)
Missing values: 0

URL Length:
count    50000.000000
mean        72.385700
std         23.858806
min         31.000000
25%         51.000000
50%         72.000000
75%         89.000000
max        139.000000
Name: URLLength, dtype: float64

Digits:
count    50000.000000
mean        13.726940
std         13.246372
min          0.000000
25%          2.000000
50%          8.000000
75%         22.000000
max         50.000000
Name: NoOfDegitsInURL, dtype: float64

Obfuscated characters:
count    50000.00000
mean         1.95810
std          2.88665
min          0.00000
25%          0.00000
50%          0.00000
75%          6.00000
max          9.00000
Name: NoOfObfuscatedChar, dtype: float64

Dots:
count    50000.000000
mean         2.665120
std          1.522124
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max          8.000000
Name: NoOfDots, dtype: float64

PROBLEM URL VS V3 COVERAGE

URLLength
Problem: 10

In [ ]:
# =========================================================
# BUILD V3 FINAL TRAINING DATA
# Original + 100k Path Augmentation + 50k Better Query Augmentation
# =========================================================

final_train_df_v3 = pd.concat(
    [
        domain_train_df,
        augmented_features_df,
        better_query_features_df,
    ],
    ignore_index=True,
)


X_train_final_v3 = final_train_df_v3[
    ROBUSTNESS_FEATURES
].copy()

y_train_final_v3 = final_train_df_v3[
    "label"
].copy()


# SAME untouched holdout
X_holdout_v3 = domain_holdout_df[
    ROBUSTNESS_FEATURES
].copy()

y_holdout_v3 = domain_holdout_df[
    "label"
].copy()


print(
    "V3 training rows:",
    len(final_train_df_v3)
)

print(
    "Holdout rows:",
    len(domain_holdout_df)
)

print(
    "Feature count:",
    X_train_final_v3.shape[1]
)

print(
    "Missing training values:",
    X_train_final_v3.isnull().sum().sum()
)

print("\nV3 training labels:")
print(
    y_train_final_v3.value_counts()
)

V3 training rows: 337856
Holdout rows: 47514
Feature count: 17
Missing training values: 0

V3 training labels:
label
1    257828
0     80028
Name: count, dtype: int64


In [ ]:
# =========================================================
# TRAIN V3 CANDIDATE MODEL
# DO NOT SAVE TO PRODUCTION YET
# =========================================================

categorical_features_v3 = ["TLD"]

numeric_features_v3 = [
    feature
    for feature in ROBUSTNESS_FEATURES
    if feature != "TLD"
]

preprocessor_v3 = ColumnTransformer(
    transformers=[
        (
            "tld",
            TargetEncoder(
                target_type="binary",
                random_state=42,
            ),
            categorical_features_v3,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features_v3,
        ),
    ]
)

candidate_model_v3 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

structural_candidate_pipeline_v3 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v3),
        ("model", candidate_model_v3),
    ]
)

print("Training V3 candidate...")

structural_candidate_pipeline_v3.fit(
    X_train_final_v3,
    y_train_final_v3,
)

print("V3 candidate training complete.")

Training V3 candidate...


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V3 candidate training complete.


In [ ]:
# =========================================================
# V3 - HOLDOUT EVALUATION
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

y_pred_v3 = structural_candidate_pipeline_v3.predict(
    X_holdout_v3
)

accuracy_v3 = accuracy_score(
    y_holdout_v3,
    y_pred_v3,
)

print(
    "V3 Holdout Accuracy:",
    round(accuracy_v3, 4)
)

print("\nClassification Report:\n")
print(
    classification_report(
        y_holdout_v3,
        y_pred_v3,
        digits=4,
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_holdout_v3,
        y_pred_v3,
    )
)

V3 Holdout Accuracy: 0.9858

Classification Report:

              precision    recall  f1-score   support

           0     0.9955    0.9714    0.9833     20492
           1     0.9787    0.9967    0.9876     27022

    accuracy                         0.9858     47514
   macro avg     0.9871    0.9840    0.9855     47514
weighted avg     0.9859    0.9858    0.9858     47514


Confusion Matrix:
[[19905   587]
 [   89 26933]]


In [ ]:
# =========================================================
# V3 - REAL LONG URL TEST
# =========================================================

real_long_url = (
    "https://www.google.com/maps/"
    "@19.1922063,72.9234965,11.82z"
    "?entry=ttu"
    "&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D"
)

real_long_features = pd.DataFrame(
    [
        extract_url_features(
            real_long_url
        )
    ]
)[ROBUSTNESS_FEATURES]


prediction = structural_candidate_pipeline_v3.predict(
    real_long_features
)[0]

probabilities = structural_candidate_pipeline_v3.predict_proba(
    real_long_features
)[0]

classes = structural_candidate_pipeline_v3.classes_

probability_map = dict(
    zip(
        classes,
        probabilities,
    )
)


print("URL:")
print(real_long_url)

print("\nPrediction:")
print(
    "Legitimate"
    if prediction == 1
    else "Phishing"
)

print(
    "\nLegitimate probability:",
    f"{probability_map.get(1, 0) * 100:.2f}%"
)

print(
    "Phishing probability:",
    f"{probability_map.get(0, 0) * 100:.2f}%"
)

URL:
https://www.google.com/maps/@19.1922063,72.9234965,11.82z?entry=ttu&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D

Prediction:
Legitimate

Legitimate probability: 85.33%
Phishing probability: 14.67%


In [ ]:
# =========================================================
# V3 - PHISHING STRESS TEST
# =========================================================

phishing_test_urls = [
    "https://paypal-login-security.com",
    "https://account-verification-update.com/login",
    "https://secure-bank-login.xyz/account",
    "https://verify-your-account.com/update",
    "https://login-security-check.net/verify",

    # Query-heavy phishing-style URLs
    "https://paypal-login-security.com/account?id=12345",
    "https://secure-bank-login.xyz/login?session=1234567890",
    "https://verify-your-account.com/update?user=12345&token=abcdef",
    "https://account-verification-update.com/login?redirect=%2Faccount%2Fverify",
    "https://login-security-check.net/verify?id=12345678901234567890&source=web",
]

print("=" * 90)
print("V3 PHISHING STRESS TEST")
print("=" * 90)

for url in phishing_test_urls:

    features = pd.DataFrame(
        [extract_url_features(url)]
    )[ROBUSTNESS_FEATURES]

    prediction = structural_candidate_pipeline_v3.predict(
        features
    )[0]

    probabilities = structural_candidate_pipeline_v3.predict_proba(
        features
    )[0]

    probability_map = dict(
        zip(
            structural_candidate_pipeline_v3.classes_,
            probabilities,
        )
    )

    phishing_prob = probability_map.get(0, 0) * 100
    legit_prob = probability_map.get(1, 0) * 100

    print("\nURL:", url)

    print(
        "Prediction:",
        "Phishing"
        if prediction == 0
        else "Legitimate"
    )

    print(
        f"Phishing: {phishing_prob:.2f}%"
        f" | Legitimate: {legit_prob:.2f}%"
    )
    

V3 PHISHING STRESS TEST

URL: https://paypal-login-security.com
Prediction: Phishing
Phishing: 97.33% | Legitimate: 2.67%

URL: https://account-verification-update.com/login
Prediction: Phishing
Phishing: 65.00% | Legitimate: 35.00%

URL: https://secure-bank-login.xyz/account
Prediction: Phishing
Phishing: 94.33% | Legitimate: 5.67%

URL: https://verify-your-account.com/update
Prediction: Phishing
Phishing: 79.16% | Legitimate: 20.84%

URL: https://login-security-check.net/verify
Prediction: Phishing
Phishing: 77.60% | Legitimate: 22.40%

URL: https://paypal-login-security.com/account?id=12345
Prediction: Legitimate
Phishing: 34.33% | Legitimate: 65.67%

URL: https://secure-bank-login.xyz/login?session=1234567890
Prediction: Phishing
Phishing: 91.33% | Legitimate: 8.67%

URL: https://verify-your-account.com/update?user=12345&token=abcdef
Prediction: Legitimate
Phishing: 48.67% | Legitimate: 51.33%

URL: https://account-verification-update.com/login?redirect=%2Faccount%2Fverify
Predicti

In [ ]:
# =========================================================
# V4 - GET TRAINING PHISHING URLS
# =========================================================

phishing_train_urls = (
    domain_train_df.loc[
        domain_train_df["label"] == 0,
        "URL"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print(
    "Training phishing URLs:",
    len(phishing_train_urls)
)

print("\nExamples:")

for url in phishing_train_urls[:10]:
    print(url)

Training phishing URLs: 80028

Examples:
http://www.teramill.com
http://www.shprakserf.gq
https://liuy-9a930.web.app/
https://ipfs.io/ipfs/qmrvvyr84esa2assw9vvwupqjgsdn4c3dwkusfdwzdz3kn?clientid=noc@protocol.ai
http://att-103731-107123.weeblysite.com/
http://www.ooguy.com
http://www.fairytalesinc.com
http://www.iuhjn.pplink.club
https://mechinchem-5cb8a.web.app/
https://fb-restriction-case-97be5.web.app/


In [ ]:
# =========================================================
# V4 - GENERATE QUERY-HEAVY PHISHING AUGMENTATION
# =========================================================

import random
from urllib.parse import urlsplit, urlunsplit

random.seed(42)

PHISHING_QUERY_SUFFIXES = [
    "?id=12345",
    "?session=1234567890",
    "?user=12345&token=abcdef",
    "?source=web&lang=en",

    "?redirect=%2Faccount%2Fverify",
    "?next=%2Flogin%2Fverify",

    "?id=12345678901234567890&source=web",

    "?token=ABCDEF12345678901234567890",

    "?data=ExampleLongValue1234567890%3D%3D",

    "?session=ABCDEF12345678901234567890"
    "&redirect=%2Faccount%2Fverify",

    "?lat=19.1922063&lng=72.9234965",

    "?entry=web"
    "&data=AbCdEf12345678901234567890%3D%3D",
]


phishing_query_pool = []

for original_url in phishing_train_urls:

    try:
        parsed = urlsplit(original_url)

        if not parsed.scheme or not parsed.netloc:
            continue

        # Keep the original phishing domain
        # and preserve its existing path when available
        path = parsed.path

        if not path or path == "/":
            path = "/login"

        base_url = urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                path,
                "",
                "",
            )
        )

        for suffix in PHISHING_QUERY_SUFFIXES:
            phishing_query_pool.append(
                base_url + suffix
            )

    except Exception:
        continue


print(
    "Phishing query-heavy pool:",
    len(phishing_query_pool)
)


PHISHING_QUERY_SAMPLE_SIZE = 50_000

phishing_query_sample = random.sample(
    phishing_query_pool,
    k=min(
        PHISHING_QUERY_SAMPLE_SIZE,
        len(phishing_query_pool),
    ),
)


print(
    "Phishing query sample:",
    len(phishing_query_sample)
)

print("\nExamples:")

for url in phishing_query_sample[:10]:
    print(url)

Phishing query-heavy pool: 960336
Phishing query sample: 50000

Examples:
http://www.nolanblog.com/login?entry=web&data=AbCdEf12345678901234567890%3D%3D
https://ccxcdff344343dfvccvn.godaddysites.com/login?source=web&lang=en
https://football.abenterprises.com.au/www/mgoqh00jxqnewrd1035qk4kv02ix/dashboad/otp.html?next=%2Flogin%2Fverify
https://fnb0-1491f.firebaseapp.com/login?data=ExampleLongValue1234567890%3D%3D
https://www.attemplate.com/nam/1c98f461-8658-4ef0-a1ed-1ff2b354dad2/888caddd-93af-45b2-80d2-6957c0ff08c8/1f5930cf-6f9a-46b8-ad1e-f1f040d59031/login?next=%2Flogin%2Fverify
https://ionostuedaiy008.firebaseapp.com/login?entry=web&data=AbCdEf12345678901234567890%3D%3D
http://www.kinfdcxv.cf/login?next=%2Flogin%2Fverify
http://www.linkundlink.de/login?id=12345
https://settings40393400390.firebaseapp.com/login?lat=19.1922063&lng=72.9234965
https://id100596210962092061.firebaseapp.com/login?session=1234567890


In [ ]:
# =========================================================
# V4 - EXTRACT PHISHING QUERY FEATURES
# =========================================================

phishing_query_features_df = pd.DataFrame(
    [
        extract_url_features(url)
        for url in phishing_query_sample
    ]
)

phishing_query_features_df.insert(
    0,
    "URL",
    phishing_query_sample
)

phishing_query_features_df["label"] = 0


print(
    "Phishing query augmentation shape:",
    phishing_query_features_df.shape
)

print(
    "Feature count:",
    phishing_query_features_df[
        ROBUSTNESS_FEATURES
    ].shape[1]
)

print(
    "Missing values:",
    phishing_query_features_df[
        ROBUSTNESS_FEATURES
    ].isnull().sum().sum()
)

print("\nLabel counts:")
print(
    phishing_query_features_df[
        "label"
    ].value_counts()
)

print("\nURL Length:")
print(
    phishing_query_features_df[
        "URLLength"
    ].describe()
)

print("\nDigits:")
print(
    phishing_query_features_df[
        "NoOfDegitsInURL"
    ].describe()
)

print("\nObfuscated characters:")
print(
    phishing_query_features_df[
        "NoOfObfuscatedChar"
    ].describe()
)

Phishing query augmentation shape: (50000, 19)
Feature count: 17
Missing values: 0

Label counts:
label
0    50000
Name: count, dtype: int64

URL Length:
count    50000.000000
mean        75.634340
std         44.536859
min         26.000000
25%         59.000000
50%         70.000000
75%         87.000000
max       4309.000000
Name: URLLength, dtype: float64

Digits:
count    50000.000000
mean        14.807540
std         19.825949
min          0.000000
25%          5.000000
50%         14.000000
75%         22.000000
max       2013.000000
Name: NoOfDegitsInURL, dtype: float64

Obfuscated characters:
count    50000.000000
mean         2.552700
std          5.060037
min          0.000000
25%          0.000000
50%          0.000000
75%          6.000000
max        453.000000
Name: NoOfObfuscatedChar, dtype: float64


In [ ]:
# =========================================================
# BUILD V4 FINAL TRAINING DATA
# Original
# + 100k Legitimate Path Augmentation
# + 50k Legitimate Query Augmentation
# + 50k Phishing Query Augmentation
# =========================================================

final_train_df_v4 = pd.concat(
    [
        domain_train_df,
        augmented_features_df,
        better_query_features_df,
        phishing_query_features_df,
    ],
    ignore_index=True,
)


X_train_final_v4 = final_train_df_v4[
    ROBUSTNESS_FEATURES
].copy()

y_train_final_v4 = final_train_df_v4[
    "label"
].copy()


# SAME untouched holdout
X_holdout_v4 = domain_holdout_df[
    ROBUSTNESS_FEATURES
].copy()

y_holdout_v4 = domain_holdout_df[
    "label"
].copy()


print(
    "V4 training rows:",
    len(final_train_df_v4)
)

print(
    "Holdout rows:",
    len(domain_holdout_df)
)

print(
    "Feature count:",
    X_train_final_v4.shape[1]
)

print(
    "Missing training values:",
    X_train_final_v4.isnull().sum().sum()
)

print("\nV4 training labels:")
print(
    y_train_final_v4.value_counts()
)

V4 training rows: 387856
Holdout rows: 47514
Feature count: 17
Missing training values: 0

V4 training labels:
label
1    257828
0    130028
Name: count, dtype: int64


In [ ]:
# =========================================================
# TRAIN V4 CANDIDATE MODEL
# =========================================================

categorical_features_v4 = ["TLD"]

numeric_features_v4 = [
    feature
    for feature in ROBUSTNESS_FEATURES
    if feature != "TLD"
]


preprocessor_v4 = ColumnTransformer(
    transformers=[
        (
            "tld",
            TargetEncoder(
                target_type="binary",
                random_state=42,
            ),
            categorical_features_v4,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features_v4,
        ),
    ]
)


candidate_model_v4 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)


structural_candidate_pipeline_v4 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v4),
        ("model", candidate_model_v4),
    ]
)


print("Training V4 candidate...")

structural_candidate_pipeline_v4.fit(
    X_train_final_v4,
    y_train_final_v4,
)

print("V4 candidate training complete.")

Training V4 candidate...


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V4 candidate training complete.


In [ ]:
# =========================================================
# V4 - COMBINED EVALUATION
# Holdout + Legit Paths + Real Long URL + Phishing Stress
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

print("=" * 90)
print("1) UNTOUCHED HOLDOUT")
print("=" * 90)

y_pred_v4 = structural_candidate_pipeline_v4.predict(
    X_holdout_v4
)

accuracy_v4 = accuracy_score(
    y_holdout_v4,
    y_pred_v4,
)

print(
    "V4 Holdout Accuracy:",
    round(accuracy_v4, 4)
)

print("\nClassification Report:\n")

print(
    classification_report(
        y_holdout_v4,
        y_pred_v4,
        digits=4,
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        y_holdout_v4,
        y_pred_v4,
    )
)


# =========================================================
# HELPER FUNCTION
# =========================================================

def test_v4_url(url):

    features = pd.DataFrame(
        [extract_url_features(url)]
    )[ROBUSTNESS_FEATURES]

    prediction = structural_candidate_pipeline_v4.predict(
        features
    )[0]

    probabilities = structural_candidate_pipeline_v4.predict_proba(
        features
    )[0]

    probability_map = dict(
        zip(
            structural_candidate_pipeline_v4.classes_,
            probabilities,
        )
    )

    phishing_prob = probability_map.get(0, 0) * 100
    legit_prob = probability_map.get(1, 0) * 100

    label = (
        "Legitimate"
        if prediction == 1
        else "Phishing"
    )

    print("\nURL:", url)
    print("Prediction:", label)
    print(
        f"Legitimate: {legit_prob:.2f}%"
        f" | Phishing: {phishing_prob:.2f}%"
    )


# =========================================================
# 2) OLD LEGITIMATE PATH TESTS
# =========================================================

print("\n" + "=" * 90)
print("2) LEGITIMATE PATH TESTS")
print("=" * 90)

legitimate_test_urls = [
    "https://www.google.com",
    "https://www.google.com/maps",
    "https://www.google.com/maps/place",
    "https://www.google.com/a/b",
    "https://www.google.com/a/b/c",
    "https://www.paypal.com",
    "https://www.paypal.com/home",
    "https://www.paypal.com/in/home",
]

for url in legitimate_test_urls:
    test_v4_url(url)


# =========================================================
# 3) REAL LONG LEGITIMATE URL
# =========================================================

print("\n" + "=" * 90)
print("3) REAL LONG LEGITIMATE URL")
print("=" * 90)

real_long_url = (
    "https://www.google.com/maps/"
    "@19.1922063,72.9234965,11.82z"
    "?entry=ttu"
    "&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D"
)

test_v4_url(
    real_long_url
)


# =========================================================
# 4) PHISHING STRESS TESTS
# =========================================================

print("\n" + "=" * 90)
print("4) PHISHING STRESS TESTS")
print("=" * 90)

phishing_test_urls = [
    "https://paypal-login-security.com",
    "https://account-verification-update.com/login",
    "https://secure-bank-login.xyz/account",
    "https://verify-your-account.com/update",
    "https://login-security-check.net/verify",

    "https://paypal-login-security.com/account?id=12345",

    "https://secure-bank-login.xyz/login?session=1234567890",

    "https://verify-your-account.com/update?user=12345&token=abcdef",

    "https://account-verification-update.com/login?redirect=%2Faccount%2Fverify",

    "https://login-security-check.net/verify?id=12345678901234567890&source=web",
]

for url in phishing_test_urls:
    test_v4_url(url)

1) UNTOUCHED HOLDOUT
V4 Holdout Accuracy: 0.9832

Classification Report:

              precision    recall  f1-score   support

           0     0.9964    0.9644    0.9802     20492
           1     0.9737    0.9974    0.9854     27022

    accuracy                         0.9832     47514
   macro avg     0.9850    0.9809    0.9828     47514
weighted avg     0.9835    0.9832    0.9831     47514

Confusion Matrix:
[[19763   729]
 [   71 26951]]

2) LEGITIMATE PATH TESTS

URL: https://www.google.com
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/maps
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/maps/place
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://www.google.com/a/b
Prediction: Legitimate
Legitimate: 74.67% | Phishing: 25.33%

URL: https://www.google.com/a/b/c
Prediction: Legitimate
Legitimate: 66.67% | Phishing: 33.33%

URL: https://www.paypal.com
Prediction: Legitim

In [ ]:
long_legitimate_tests = [
    "https://example.com/search?q=hello%20world&page=2",
    "https://example.com/location?lat=19.1922063&lng=72.9234965",
    "https://example.com/docs/view?id=12345&lang=en",

    "https://example.com/location/@19.1922063,72.9234965,11.82z?entry=web&mode=view",

    "https://example.com/location/@19.1922063,72.9234965,11.82z?entry=web&data=ExampleLongValue1234567890%3D%3D",

    "https://example.com/docs/view?id=12345678901234567890&source=browser&lang=en",

    "https://example.com/redirect?next=%2Faccount%2Fsettings%2Fprivacy&source=web",

    "https://example.com/place/@51.507351,-0.127758,14.25z?entry=browser&data=Example1234567890%3D%3D",
]

print("LONG LEGITIMATE TESTS")

for url in long_legitimate_tests:
    test_v4_url(url)

LONG LEGITIMATE TESTS

URL: https://example.com/search?q=hello%20world&page=2
Prediction: Legitimate
Legitimate: 85.33% | Phishing: 14.67%

URL: https://example.com/location?lat=19.1922063&lng=72.9234965
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://example.com/docs/view?id=12345&lang=en
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://example.com/location/@19.1922063,72.9234965,11.82z?entry=web&mode=view
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://example.com/location/@19.1922063,72.9234965,11.82z?entry=web&data=ExampleLongValue1234567890%3D%3D
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://example.com/docs/view?id=12345678901234567890&source=browser&lang=en
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https://example.com/redirect?next=%2Faccount%2Fsettings%2Fprivacy&source=web
Prediction: Legitimate
Legitimate: 100.00% | Phishing: 0.00%

URL: https:/

In [ ]:
# =========================================================
# SAVE FINAL V4 PRODUCTION MODEL
# =========================================================

import joblib
from pathlib import Path

production_model_path = Path(
    "../models/trustlens_production_pipeline.joblib"
)

joblib.dump(
    structural_candidate_pipeline_v4,
    production_model_path
)

print(
    "Saved V4 production model to:",
    production_model_path
)

NameError: name 'structural_candidate_pipeline_v4' is not defined

In [ ]:
# =========================================================
# SAVE FINAL V4 PRODUCTION MODEL
# =========================================================

import joblib
from pathlib import Path

production_model_path = Path(
    "../models/trustlens_production_pipeline.joblib"
)

joblib.dump(
    structural_candidate_pipeline_v4,
    production_model_path
)

print("Saved V4 production model to:", production_model_path)

NameError: name 'structural_candidate_pipeline_v4' is not defined

In [ ]:
ROBUSTNESS_FEATURES = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLD",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "NoOfDegitsInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "IsHTTPS",
    "NoOfDots",
    "NoOfSlashes",
    "SuspiciousKeywordCount",
    "HasHyphenInDomain",
    "PathDepth",
]

print("ROBUSTNESS_FEATURES restored:", len(ROBUSTNESS_FEATURES))

ROBUSTNESS_FEATURES restored: 17


In [ ]:
# =========================================================
# RECOVERY STEP 2 - REBUILD V4 TRAIN / HOLDOUT MATRICES
# =========================================================

final_train_df_v4 = pd.concat(
    [
        domain_train_df,
        augmented_features_df,
        better_query_features_df,
        phishing_query_features_df,
    ],
    ignore_index=True,
)

X_train_final_v4 = final_train_df_v4[
    ROBUSTNESS_FEATURES
].copy()

y_train_final_v4 = final_train_df_v4[
    "label"
].copy()

X_holdout_v4 = domain_holdout_df[
    ROBUSTNESS_FEATURES
].copy()

y_holdout_v4 = domain_holdout_df[
    "label"
].copy()

print("V4 training rows:", len(final_train_df_v4))
print("Holdout rows:", len(domain_holdout_df))
print("Feature count:", X_train_final_v4.shape[1])
print("Missing values:", X_train_final_v4.isnull().sum().sum())

print("\nV4 label counts:")
print(y_train_final_v4.value_counts())

NameError: name 'better_query_features_df' is not defined

In [ ]:
required_vars = [
    "domain_train_df",
    "domain_holdout_df",
    "augmented_features_df",
    "better_query_features_df",
    "phishing_query_features_df",
    "extract_url_features",
]

for name in required_vars:
    print(f"{name}: {'EXISTS' if name in globals() else 'MISSING'}")

domain_train_df: EXISTS
domain_holdout_df: EXISTS
augmented_features_df: EXISTS
better_query_features_df: MISSING
phishing_query_features_df: MISSING
extract_url_features: EXISTS


In [ ]:
# =========================================================
# RECOVERY STEP 3
# Recreate V3 legitimate query augmentation
# + V4 phishing query augmentation
# =========================================================

import random
import pandas as pd
from urllib.parse import urlparse, urlunparse

# ---------------------------------------------------------
# A) RECREATE BETTER LEGITIMATE QUERY AUGMENTATION (V3)
# ---------------------------------------------------------

legitimate_train_urls = (
    domain_train_df[
        domain_train_df["label"] == 1
    ]["URL"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

NORMAL_QUERY_TEMPLATES = [
    "/search?q=example",
    "/search?q=example&page=2",
    "/products?id=12345",
    "/article?id=987654",
    "/account/settings?tab=privacy",
    "/docs/view?id=12345&lang=en",
    "/results?query=example&sort=recent",
    "/location?lat=19.1922063&lng=72.9234965",
    "/search?q=hello%20world",
    "/redirect?next=%2Faccount%2Fsettings",
]

COMPLEX_QUERY_TEMPLATES = [
    "/location/@19.1922063,72.9234965,11.82z?entry=web&mode=view",
    "/location/@19.1922063,72.9234965,11.82z?entry=web&source=browser&mode=view",
    "/location/@19.1922063,72.9234965,11.82z?entry=web&data=ExampleLongValue1234567890%3D%3D",
    "/location/@19.1922063,72.9234965,11.82z?source=browser&data=AbCdEf12345678901234567890%3D%3D",
    "/view/item/1234567890?session=ABCDEF12345678901234567890&lang=en",
    "/docs/view?id=12345678901234567890&source=browser&lang=en",
    "/redirect?next=%2Faccount%2Fsettings%2Fprivacy&source=web",
    "/search?q=hello%20world&category=articles&page=123456",
    "/data/@40.712776,-74.005974,12.50z?view=detail&source=web",
    "/place/@51.507351,-0.127758,14.25z?entry=browser&data=Example1234567890%3D%3D",
]

normal_query_pool = []
complex_query_pool = []

for url in legitimate_train_urls:
    parsed = urlparse(url)

    scheme = parsed.scheme if parsed.scheme else "https"
    netloc = parsed.netloc

    if not netloc:
        continue

    for template in NORMAL_QUERY_TEMPLATES:
        normal_query_pool.append(
            f"{scheme}://{netloc}{template}"
        )

    for template in COMPLEX_QUERY_TEMPLATES:
        complex_query_pool.append(
            f"{scheme}://{netloc}{template}"
        )

random.seed(42)

sampled_normal_urls = random.sample(
    normal_query_pool,
    25000
)

sampled_complex_urls = random.sample(
    complex_query_pool,
    25000
)

better_query_urls = (
    sampled_normal_urls
    + sampled_complex_urls
)

better_query_features_df = pd.DataFrame(
    [
        {
            **extract_url_features(url),
            "URL": url,
            "label": 1,
        }
        for url in better_query_urls
    ]
)


# ---------------------------------------------------------
# B) RECREATE PHISHING QUERY AUGMENTATION (V4)
# ---------------------------------------------------------

phishing_train_urls = (
    domain_train_df[
        domain_train_df["label"] == 0
    ]["URL"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

PHISHING_QUERY_SUFFIXES = [
    "?id=12345",
    "?session=1234567890",
    "?user=12345&token=abcdef",
    "?source=web&lang=en",
    "?redirect=%2Faccount%2Fverify",
    "?next=%2Flogin%2Fverify",
    "?id=12345678901234567890&source=web",
    "?token=ABCDEF12345678901234567890",
    "?data=ExampleLongValue1234567890%3D%3D",
    "?session=ABCDEF12345678901234567890&redirect=%2Faccount%2Fverify",
    "?lat=19.1922063&lng=72.9234965",
    "?entry=web&data=AbCdEf12345678901234567890%3D%3D",
]

phishing_query_pool = []

for url in phishing_train_urls:
    parsed = urlparse(url)

    scheme = parsed.scheme if parsed.scheme else "https"
    netloc = parsed.netloc

    if not netloc:
        continue

    path = parsed.path

    if not path or path == "/":
        path = "/login"

    base_url = urlunparse(
        (
            scheme,
            netloc,
            path,
            "",
            "",
            "",
        )
    )

    for suffix in PHISHING_QUERY_SUFFIXES:
        phishing_query_pool.append(
            base_url + suffix
        )

random.seed(42)

sampled_phishing_query_urls = random.sample(
    phishing_query_pool,
    50000
)

phishing_query_features_df = pd.DataFrame(
    [
        {
            **extract_url_features(url),
            "URL": url,
            "label": 0,
        }
        for url in sampled_phishing_query_urls
    ]
)


# ---------------------------------------------------------
# VERIFY
# ---------------------------------------------------------

print("better_query_features_df:", better_query_features_df.shape)
print("phishing_query_features_df:", phishing_query_features_df.shape)

print("\nLegitimate query labels:")
print(better_query_features_df["label"].value_counts())

print("\nPhishing query labels:")
print(phishing_query_features_df["label"].value_counts())

better_query_features_df: (50000, 19)
phishing_query_features_df: (50000, 19)

Legitimate query labels:
label
1    50000
Name: count, dtype: int64

Phishing query labels:
label
0    50000
Name: count, dtype: int64


In [ ]:
# =========================================================
# RECOVERY STEP 4 - REBUILD V4 TRAIN / HOLDOUT MATRICES
# =========================================================

final_train_df_v4 = pd.concat(
    [
        domain_train_df,
        augmented_features_df,
        better_query_features_df,
        phishing_query_features_df,
    ],
    ignore_index=True,
)

X_train_final_v4 = final_train_df_v4[
    ROBUSTNESS_FEATURES
].copy()

y_train_final_v4 = final_train_df_v4[
    "label"
].copy()

X_holdout_v4 = domain_holdout_df[
    ROBUSTNESS_FEATURES
].copy()

y_holdout_v4 = domain_holdout_df[
    "label"
].copy()

print("V4 training rows:", len(final_train_df_v4))
print("Holdout rows:", len(domain_holdout_df))
print("Feature count:", X_train_final_v4.shape[1])
print("Missing values:", X_train_final_v4.isnull().sum().sum())

print("\nV4 label counts:")
print(y_train_final_v4.value_counts())

V4 training rows: 387856
Holdout rows: 47514
Feature count: 17
Missing values: 0

V4 label counts:
label
1    257828
0    130028
Name: count, dtype: int64


In [ ]:
# =========================================================
# RECOVERY STEP 5 - DEFINE + TRAIN V4 CANDIDATE
# =========================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

categorical_features_v4 = ["TLD"]

numeric_features_v4 = [
    feature
    for feature in ROBUSTNESS_FEATURES
    if feature != "TLD"
]

preprocessor_v4 = ColumnTransformer(
    transformers=[
        (
            "tld",
            TargetEncoder(
                target_type="binary",
                random_state=42,
            ),
            categorical_features_v4,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features_v4,
        ),
    ]
)

candidate_model_v4 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

structural_candidate_pipeline_v4 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v4),
        ("model", candidate_model_v4),
    ]
)

print("Training V4 candidate...")

structural_candidate_pipeline_v4.fit(
    X_train_final_v4,
    y_train_final_v4,
)

print("V4 candidate training complete.")

Training V4 candidate...


c:\Users\swana\OneDrive\Desktop\TrustLens\venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


V4 candidate training complete.


In [ ]:
# =========================================================
# RECOVERY STEP 6 - SAVE + RELOAD FINAL V4 MODEL
# =========================================================

import joblib
from pathlib import Path
import pandas as pd

production_model_path = Path(
    "../models/trustlens_production_pipeline.joblib"
)

# Save V4
joblib.dump(
    structural_candidate_pipeline_v4,
    production_model_path
)

print("Saved V4 production model to:", production_model_path)

# Reload from disk
reloaded_v4 = joblib.load(
    production_model_path
)

print("Reload successful.")
print("Classes:", reloaded_v4.classes_)

# ---------------------------------------------------------
# VERIFY REAL LONG LEGITIMATE URL
# ---------------------------------------------------------

real_long_url = (
    "https://www.google.com/maps/"
    "@19.1922063,72.9234965,11.82z"
    "?entry=ttu"
    "&g_ep=EgoyMDI2MDkwMi4wIKXMDSoASAFQAw%3D%3D"
)

features = pd.DataFrame(
    [extract_url_features(real_long_url)]
)[ROBUSTNESS_FEATURES]

prediction = reloaded_v4.predict(features)[0]
probabilities = reloaded_v4.predict_proba(features)[0]

probability_map = dict(
    zip(
        reloaded_v4.classes_,
        probabilities,
    )
)

print("\nREAL LONG URL TEST")
print(
    "Prediction:",
    "Legitimate" if prediction == 1 else "Phishing"
)
print(
    f"Legitimate: {probability_map.get(1, 0) * 100:.2f}%"
    f" | Phishing: {probability_map.get(0, 0) * 100:.2f}%"
)

# ---------------------------------------------------------
# VERIFY PHISHING QUERY URL
# ---------------------------------------------------------

phishing_test_url = (
    "https://account-verification-update.com/"
    "login?redirect=%2Faccount%2Fverify"
)

features = pd.DataFrame(
    [extract_url_features(phishing_test_url)]
)[ROBUSTNESS_FEATURES]

prediction = reloaded_v4.predict(features)[0]
probabilities = reloaded_v4.predict_proba(features)[0]

probability_map = dict(
    zip(
        reloaded_v4.classes_,
        probabilities,
    )
)

print("\nPHISHING QUERY TEST")
print(
    "Prediction:",
    "Legitimate" if prediction == 1 else "Phishing"
)
print(
    f"Legitimate: {probability_map.get(1, 0) * 100:.2f}%"
    f" | Phishing: {probability_map.get(0, 0) * 100:.2f}%"
)

Saved V4 production model to: ..\models\trustlens_production_pipeline.joblib
Reload successful.
Classes: [0 1]

REAL LONG URL TEST
Prediction: Legitimate
Legitimate: 79.67% | Phishing: 20.33%

PHISHING QUERY TEST
Prediction: Phishing
Legitimate: 0.00% | Phishing: 100.00%


In [ ]:
# =========================================================
# RECOVERY STEP 7 - VERIFY SAVED MODEL + FEATURES
# =========================================================

print("MODEL OBJECT")
print(type(reloaded_v4))

print("\nMODEL CLASSES")
print(reloaded_v4.classes_)

print("\nROBUSTNESS FEATURES")
print(ROBUSTNESS_FEATURES)

print("\nEXTRACTED FEATURES FOR GOOGLE MAPS")
google_features = extract_url_features(real_long_url)

for feature in ROBUSTNESS_FEATURES:
    print(f"{feature}: {google_features[feature]}")

print("\nFINAL PREDICTION")
google_df = pd.DataFrame([google_features])[ROBUSTNESS_FEATURES]

print("Prediction:", reloaded_v4.predict(google_df)[0])
print("Probabilities:", reloaded_v4.predict_proba(google_df)[0])

MODEL OBJECT


NameError: name 'reloaded_v4' is not defined